### RAG with PDF 📄 Data extraction to give context to LLM 🧠

<img src="./Images/RAGs.png" width="800" height="700" style="display: block; margin: auto;">

In [ ]:
!pip install pypdf

In [ ]:
from dotenv import load_dotenv
load = load_dotenv('./../.env')


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

### 1. Extracting the PDF files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(len(documents))

In [ ]:
print(documents[:1])

### 2. Text Splitting into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                               chunk_overlap=200, add_start_index=True)

all_splits = text_splitter.split_documents(documents)

len(all_splits)

### 3. Embedding

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="llama3.2:latest")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
len(vector_1), len(vector_1)

### 4. Vector Stores

In [ ]:
#!pip install -qU langchain-chroma

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = all_splits,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

### 5. Retriving from the Persistant Vector Datastore

In [ ]:
from langchain_chroma import Chroma


vector_store = Chroma(persist_directory='./chroma_langchain_db', embedding_function=embeddings)

result = vector_store.similarity_search("What is Bias testing", k=3)

for doc in result:
    print(doc.page_content)

In [ ]:
result = vector_store.similarity_search_with_score("What are the types of LLM Testing")

result[0]

### 6. Retrivers in Langchain

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs = {"k": 1}
)

retriever.batch(
    [
        "What is the Bias Measurement",
        "How to test human safety against LLM",
        "How LLM forgets the context"
    ]
)


### Document Retrival Manually

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

query = "What exactly does Testing the Factual Correctness of LLM tells"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt_template = ChatPromptTemplate.from_template(
    """
    You are an AI Assisant. Use the following context to answer the question correctly.
    If you dont know the answer, just tell, I dont know.
    
    Also, summarize the response in MD format
    
    "context: {context} \n\n"
    "question: {question} \n\n"
    "AI answer:
    
    """
)

chain = prompt_template | llm | StrOutputParser()

response = chain.invoke({"context": context_text, "question": query})

print(response)

### Using Langchain Hub for Prompt
#### Langchain Hub has breaking change in v1.0.0

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchainhub import Client
import json
from langchain_core.load import load

query = "How to test Translation in LLM?"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])


hub = Client()
prompt_data = hub.pull("rlm/rag-prompt")

prompt_dict = json.loads(prompt_data)
prompt = load(prompt_dict)

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"context": context_text, "question": query})

print(response)

/var/folders/jc/c7p4f2sd36xbqwkp2djn8flc0000gn/T/ipykernel_16378/3765046818.py:14: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_data = hub.pull("rlm/rag-prompt")
/Users/karthik/Downloads/LangChainTrainings-4/myenv312/lib/python3.13/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
/var/folders/jc/c7p4f2sd36xbqwkp2djn8flc0000gn/T/ipykernel_16378/3765046818.py:17: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  prompt = load(prompt_dict)


To test translation in LLMs like ChatGPT and GPT-4, you can use human evaluation on randomly selected responses where different models have differing judgments. Translate responses from other languages to English using tools like Google Translate before evaluating them with the model to ensure consistency across languages.


### Retrieving data using RetrievalQA is obsolete now. Use Chains instead.
### Breaking changes in Langchain v1.0.0

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Format documents function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain with LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Invoke
response = rag_chain.invoke("What exactly does Testing the Factual Correctness of LLM tells")
print(response)

Testing the factual correctness of LLMs involves assessing their accuracy on multi-hop questions, which are more challenging. FactChecker generates these questions and evaluates LLM responses, revealing higher error rates compared to single-hop questions. This process highlights the increased difficulty multi-hop questions pose for LLMs.
